# 第17章 图像分割与提取（跟随代码）

> 《OpenCV轻松入门：面向Python》第 17 章的跟随代码，内容未做改动（素材替换处以注释说明）。
> 运行前请确认工作目录为本文件夹（picture*.jpg 等素材都在这里）。

# 17.1 用分水岭算法实现图像分割与提取

## 17.1.2 相关函数介绍：形态学获取图像边界（例17.1）

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

o = cv2.imread("picture9.jpg", cv2.IMREAD_UNCHANGED)  # 原书 my.bmp：黑底白色图形
k = np.ones((5, 5), np.uint8)      # 5×5 全 1 结构元
e = cv2.erode(o, k)                # 腐蚀：前景整体向内收缩一圈
b = cv2.subtract(o, e)             # 原图减腐蚀图：差集就是一圈边界

plt.subplot(131)                   # 1 行 3 列的第 1 幅
plt.imshow(o)                      # 原始图像
plt.axis('off')
plt.subplot(132)
plt.imshow(e)                      # 腐蚀结果
plt.axis('off')
plt.subplot(133)
plt.imshow(b)                      # 边界图像
plt.axis('off')
plt.show()

## 距离变换获取确定前景（例17.2）

In [ ]:
img = cv2.imread('picture13.jpg')          # 原书 water_coins.jpg
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)   # 转灰度
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)     # matplotlib 按 RGB 显示，先转序
ishow = img.copy()                         # 留一份原始图用于展示

ret, thresh = cv2.threshold(gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU)   # Otsu 自动阈值。picture13 是深底亮币，用 BINARY；
                                           # 原书 water_coins 是浅底暗币，用 THRESH_BINARY_INV
closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE,
    np.ones((15, 15), np.uint8))           # 素材适配：闭运算填掉币面深色"5角"字样留下的洞
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(closing, cv2.MORPH_OPEN,
    kernel, iterations=2)                  # 开运算×2 去噪（原书对 thresh 直接开运算）

dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)   # 距离变换
ret, fore = cv2.threshold(dist_transform,
    0.7 * dist_transform.max(), 255, 0)    # 以 0.7 倍最大值做阈值化，得到确定前景

plt.subplot(131)                           # 1 行 3 列第 1 幅
plt.imshow(ishow)                          # 原始图像
plt.axis('off')
plt.subplot(132)
plt.imshow(dist_transform)                 # 距离图像
plt.axis('off')
plt.subplot(133)
plt.imshow(fore)                           # 阈值化后的确定前景
plt.axis('off')
plt.show()

## 确定背景、确定前景与未知区域（例17.3）

In [ ]:
img = cv2.imread('picture13.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)     # 转灰度
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)       # 转 RGB 供 matplotlib 显示
ishow = img.copy()                               # 原始图像副本

ret, thresh = cv2.threshold(gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU)         # Otsu 阈值分割（深底亮币用 BINARY）
closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE,
    np.ones((15, 15), np.uint8))                 # 素材适配：填掉币面字样的洞
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(closing, cv2.MORPH_OPEN,
    kernel, iterations=2)                        # 开运算去噪

bg = cv2.dilate(opening, kernel, iterations=3)   # 膨胀×3：前景放大，背景被压缩
                                                 # bg 的背景部分 = 确定背景 B
dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)   # 距离变换
ret, fore = cv2.threshold(dist, 0.7 * dist.max(), 255, 0)   # 确定前景 F
fore = np.uint8(fore)                            # 转成 uint8 方便后续运算
un = cv2.subtract(bg, fore)                      # UN = (O - B) - F

plt.subplot(221)                                 # 2 行 2 列第 1 幅
plt.imshow(ishow)                                # 原始图像
plt.axis('off')
plt.subplot(222)
plt.imshow(bg)                                   # 膨胀结果
plt.axis('off')
plt.subplot(223)
plt.imshow(fore)                                 # 确定前景
plt.axis('off')
plt.subplot(224)
plt.imshow(un)                                   # 未知区域
plt.axis('off')
plt.show()

## 连通域标注确定前景（例17.4）

In [ ]:
img = cv2.imread('picture13.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)     # 转灰度
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)       # 转 RGB 显示
ishow = img.copy()                               # 原始图像副本

ret, thresh = cv2.threshold(gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU)         # Otsu 阈值分割
closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE,
    np.ones((15, 15), np.uint8))                 # 素材适配：填洞
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(closing, cv2.MORPH_OPEN,
    kernel, iterations=2)                        # 开运算去噪

sure_bg = cv2.dilate(opening, kernel, iterations=3)      # 确定背景（本例未直接使用）
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)   # 距离变换
ret, fore = cv2.threshold(dist_transform,
    0.7 * dist_transform.max(), 255, 0)          # 确定前景
fore = np.uint8(fore)                            # 转 uint8

ret, markers = cv2.connectedComponents(fore)     # 连通域标注：背景 0，对象 1 起
print(ret)                                       # 打印标注数量（含背景）

plt.subplot(131)
plt.imshow(ishow)                                # 原始图像
plt.axis('off')
plt.subplot(132)
plt.imshow(fore)                                 # 确定前景的中心点图像
plt.axis('off')
plt.subplot(133)
plt.imshow(markers)                              # 标注结果
plt.axis('off')
plt.show()

## 标注修正，未知区域置 0（例17.5）

In [ ]:
img = cv2.imread('picture13.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)     # 转灰度
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)       # 转 RGB 显示
ishow = img.copy()

ret, thresh = cv2.threshold(gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU)         # Otsu 阈值分割
closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE,
    np.ones((15, 15), np.uint8))                 # 素材适配：填洞
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(closing, cv2.MORPH_OPEN,
    kernel, iterations=2)                        # 开运算去噪
sure_bg = cv2.dilate(opening, kernel, iterations=3)      # 确定背景
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
ret, fore = cv2.threshold(dist_transform,
    0.7 * dist_transform.max(), 255, 0)          # 确定前景
fore = np.uint8(fore)

ret, markers1 = cv2.connectedComponents(fore)    # 直接标注的结果
foreAdv = fore.copy()                            # 复制一份确定前景
unknown = cv2.subtract(sure_bg, foreAdv)         # 未知区域 = 确定背景 - 确定前景
ret, markers2 = cv2.connectedComponents(foreAdv) # 重新标注
markers2 = markers2 + 1                          # 整体加 1：1 是背景，2 起是前景
markers2[unknown == 255] = 0                     # 未知区域置 0

plt.subplot(121)                                 # 1 行 2 列第 1 幅
plt.imshow(markers1)                             # 修正前
plt.axis('off')
plt.subplot(122)
plt.imshow(markers2)                             # 修正后
plt.axis('off')
plt.show()

## 17.1.3 分水岭算法图像分割实例（例17.6）

In [ ]:
img = cv2.imread('picture13.jpg')                # 原书 water_coins.jpg
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)     # 转灰度
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)       # 转 RGB 显示
ishow = img.copy()                               # 留一份原始图

ret, thresh = cv2.threshold(gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU)         # 步骤 0：Otsu 二值化（深底亮币用 BINARY）
closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE,
    np.ones((15, 15), np.uint8))                 # 素材适配：闭运算填掉币面深色字样的洞
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(closing, cv2.MORPH_OPEN,
    kernel, iterations=2)                        # 步骤 1：开运算去噪

sure_bg = cv2.dilate(opening, kernel, iterations=3)       # 步骤 2：膨胀得到确定背景 B
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
ret, sure_fg = cv2.threshold(dist_transform,
    0.7 * dist_transform.max(), 255, 0)          # 步骤 3：距离变换 + 阈值化得到确定前景 F
sure_fg = np.uint8(sure_fg)                      # 转 uint8
unknown = cv2.subtract(sure_bg, sure_fg)         # 步骤 4：未知区域 UN = (O - B) - F

ret, markers = cv2.connectedComponents(sure_fg)  # 步骤 5：对确定前景标注
markers = markers + 1                            # 步骤 6a：整体加 1，让背景=1、前景从 2 起
markers[unknown == 255] = 0                      # 步骤 6b：未知区域置 0

markers = cv2.watershed(img, markers)            # 步骤 7：分水岭分割，边界写 -1
img[markers == -1] = [0, 255, 0]                 # 把边界涂成绿色

plt.subplot(121)                                 # 1 行 2 列第 1 幅
plt.imshow(ishow)                                # 原始图像
plt.axis('off')
plt.subplot(122)
plt.imshow(img)                                  # 分割结果：绿色为分割线
plt.axis('off')
plt.show()

# 17.2 交互式前景提取

## 矩形模式 GrabCut（例17.7）

In [ ]:
o = cv2.imread('picture3.jpg')                   # 原书 lenacolor.png（512×512）
orgb = cv2.cvtColor(o, cv2.COLOR_BGR2RGB)        # 转 RGB 显示
mask = np.zeros(o.shape[:2], np.uint8)           # 全 0 掩模：初始全部当作确定背景
bgdModel = np.zeros((1, 65), np.float64)         # 算法内部数组
fgdModel = np.zeros((1, 65), np.float64)
rect = (60, 100, 620, 850)                       # 前景大致范围 (x,y,w,h)；原书 lena 用 (50,50,400,500)
cv2.grabCut(o, mask, rect, bgdModel, fgdModel, 5,
    cv2.GC_INIT_WITH_RECT)                       # 矩形模式，迭代 5 次

mask2 = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')   # 0/2 当背景，1/3 当前景
ogc = o * mask2[:, :, np.newaxis]                # 掩模逐通道乘原图：背景清零
ogc = cv2.cvtColor(ogc, cv2.COLOR_BGR2RGB)       # 转 RGB 显示

plt.subplot(121)                                 # 1 行 2 列第 1 幅
plt.imshow(orgb)                                 # 原始图像
plt.axis('off')
plt.subplot(122)
plt.imshow(ogc)                                  # GrabCut 提取结果
plt.axis('off')
plt.show()

## 掩模模式 GrabCut（例17.8）

In [ ]:
o = cv2.imread('picture3.jpg')                   # 原书 lenacolor.png
orgb = cv2.cvtColor(o, cv2.COLOR_BGR2RGB)        # 转 RGB 显示
mask = np.zeros(o.shape[:2], np.uint8)           # 全 0 掩模
bgd = np.zeros((1, 65), np.float64)
fgd = np.zeros((1, 65), np.float64)
rect = (60, 100, 620, 850)                       # 原书 lena 用 (50,50,400,500)
cv2.grabCut(o, mask, rect, bgd, fgd, 5,
    cv2.GC_INIT_WITH_RECT)                       # 步骤 1：矩形模式初步提取

# 原书用 Windows 画笔在图上手工涂抹并另存为 mask.png：
# 白色 = 要保留的前景，黑色 = 要删除的背景。这里用程序生成等价的"手绘"模板：
# 脸部涂白（保留），左上、右上两个背景角涂黑（删除），其余涂中性灰表示不干预
m0 = np.full(o.shape[:2], 127, np.uint8)         # 中性灰 = 不打标记
cv2.rectangle(m0, (220, 320), (520, 720), 255, -1)   # 白笔刷：脸
cv2.rectangle(m0, (0, 0), (120, 160), 0, -1)         # 黑笔刷：左上背景
cv2.rectangle(m0, (616, 0), (736, 200), 0, -1)       # 黑笔刷：右上背景
cv2.imwrite('mask.png', m0)                      # 存成 mask.png，与原书流程对齐

mask2 = cv2.imread('mask.png', 0)                # 以灰度方式读入模板
mask2Show = cv2.imread('mask.png', -1)           # 原样读入用于显示
plt.subplot(121)                                 # 1 行 2 列第 1 幅
plt.imshow(mask2Show, cmap='gray')               # 模板（原书把模板转 RGB 显示）
plt.axis('off')

mask[mask2 == 0] = 0                             # 黑色（0）→ 确定背景
mask[mask2 == 255] = 1                           # 白色（255）→ 确定前景
mask, bgd, fgd = cv2.grabCut(o, mask, None, bgd, fgd, 5,
    cv2.GC_INIT_WITH_MASK)                       # 步骤 7：掩模模式继续迭代
mask = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')   # 合并四值为二值
ogc = o * mask[:, :, np.newaxis]                 # 背景清零
ogc = cv2.cvtColor(ogc, cv2.COLOR_BGR2RGB)

plt.subplot(122)
plt.imshow(ogc)                                  # 提取结果
plt.axis('off')
plt.show()

## 直接自定义模板 GrabCut（例17.9）

In [ ]:
o = cv2.imread('picture3.jpg')                   # 原书 lenacolor.png
orgb = cv2.cvtColor(o, cv2.COLOR_BGR2RGB)        # 转 RGB 显示
bgd = np.zeros((1, 65), np.float64)
fgd = np.zeros((1, 65), np.float64)
mask2 = np.zeros(o.shape[:2], np.uint8)          # 先全部构造为 0（确定背景），后续逐步细化
mask2[100:950, 60:680] = 3                       # 人像的可能前景区域；原书 lena 用 mask2[30:512,50:400]=3
mask2[320:720, 220:520] = 1                      # 人像的确定前景区域；不设置这块，头像提取不完整
                                                 # 原书 lena 用 mask2[50:300,150:200]=1
cv2.grabCut(o, mask2, None, bgd, fgd, 5,
    cv2.GC_INIT_WITH_MASK)                       # 直接掩模模式，不做矩形初始化
mask2 = np.where((mask2 == 2) | (mask2 == 0), 0, 1).astype('uint8')   # 合并四值为二值
ogc = o * mask2[:, :, np.newaxis]                # 背景清零
ogc = cv2.cvtColor(ogc, cv2.COLOR_BGR2RGB)

plt.subplot(121)
plt.imshow(orgb)                                 # 原始图像
plt.axis('off')
plt.subplot(122)
plt.imshow(ogc)                                  # 提取结果
plt.axis('off')
plt.show()

# 拓展：K-means 聚类分割

## cv2.kmeans 已知分组对照

In [ ]:
import numpy as np
import cv2

# 第 1 步：构造三组已知分组的二维点（真值：中心 (0,0)、(60,60)、(120,0)）
rng = np.random.RandomState(42)                    # 固定随机种子，结果可复现
g0 = rng.normal(loc=(0, 0), scale=2.0, size=(60, 2))     # 第 0 组 60 个点
g1 = rng.normal(loc=(60, 60), scale=2.0, size=(60, 2))   # 第 1 组 60 个点
g2 = rng.normal(loc=(120, 0), scale=2.0, size=(60, 2))   # 第 2 组 60 个点
true_labels = np.array([0] * 60 + [1] * 60 + [2] * 60)   # 已知分组标签
data = np.vstack([g0, g1, g2]).astype(np.float32)        # 合并并转 float32（cv2.kmeans 硬性要求）

# 第 2 步：设置参数并调用
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)   # 迭代 10 次或精度 1.0 即停
compactness, labels, centers = cv2.kmeans(data, 3, None, criteria, 10,
    cv2.KMEANS_RANDOM_CENTERS)                     # K=3，尝试 10 次，随机初始中心

# 第 3 步：对照已知分组（簇编号与真实分组可能整体置换，先按类心就近对齐）
true_centers = np.array([[0, 0], [60, 60], [120, 0]], np.float32)
lab2true = {k: int(np.argmin(np.linalg.norm(true_centers - centers[k], axis=1)))
            for k in range(3)}                     # 每个簇的实际中心 → 最近的已知中心
pred = np.array([lab2true[l] for l in labels.ravel()])   # 簇编号翻译回真实分组
print("compactness =", compactness)                # 紧密度（正浮点数）
print("labels.shape =", labels.shape, "dtype =", labels.dtype)   # (180, 1) int32
print("centers =", np.round(centers, 1))           # 三个类心
print("与已知分组一致率 =", (pred == true_labels).mean())   # 应为 1.0

## K-means 颜色量化分割

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('picture13.jpg')                  # 读入彩色图
Z = img.reshape((-1, 3)).astype(np.float32)        # R×C×3 摊平成 N×3，每行一个像素
K = 4                                              # 聚成 4 种颜色
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
ret, label, center = cv2.kmeans(Z, K, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

center = np.uint8(center)                          # 类心还原成 uint8 颜色
res = center[label.ravel()]                        # 每个像素替换为其类心颜色
quant = res.reshape(img.shape)                     # 还原成图像形状

plt.subplot(121)                                   # 1 行 2 列第 1 幅
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))   # 原图
plt.axis('off')
plt.subplot(122)
plt.imshow(cv2.cvtColor(quant, cv2.COLOR_BGR2RGB)) # 4 色量化结果
plt.axis('off')
plt.show()
print("compactness =", ret)                        # 本次量化的紧密度